# 🏠 01 — Exploración: Idealista API
Fuente: Idealista Property Search API v3.5  
Datos: precios de venta y alquiler en las 7 principales ciudades españolas  
Objetivo: entender la estructura del JSON, distribución de precios y cobertura geográfica

Celda 2 — Imports y carga del JSON más reciente

In [1]:
import json
import pandas as pd
import glob
import os
from pathlib import Path

ROOT = Path(os.getcwd()).parent.parent  # sube desde notebooks/01_exploracion/ a raíz
DATA_DIR = ROOT / "data" / "idealista"

# Carga el JSON más reciente disponible
archivos = sorted(glob.glob(str(DATA_DIR / "**" / "*.json"), recursive=True))
if not archivos:
    raise FileNotFoundError(f"No hay datos de Idealista en {DATA_DIR}. Ejecuta trigger.py primero.")

ultimo = archivos[-1]
print(f"📂 Cargando: {ultimo}")
with open(ultimo) as f:
    raw = json.load(f)

print(f"✅ Snapshot del: {raw.get('snapshot_date', 'desconocido')}")
print(f"📊 Peticiones usadas ese mes: {raw.get('peticiones_usadas_este_mes', '?')}/100")

📂 Cargando: /workspaces/TFG_Spain-s_Migratory_Flow/data/idealista/2026/06/20_06_2026.json
✅ Snapshot del: 2026-06-20
📊 Peticiones usadas ese mes: 30/100


Celda 3 — Convertir a DataFrame

In [2]:
registros = raw.get("anuncios", [])
df = pd.DataFrame(registros)

print(f"Total anuncios: {len(df)}")
print(f"Columnas: {list(df.columns)}")
df.head(3)

Total anuncios: 700
Columnas: ['snapshot_date', 'city', 'operation', 'propertyCode', 'price', 'size_m2', 'price_m2', 'rooms', 'bathrooms', 'floor', 'exterior', 'district', 'municipality', 'neighborhood', 'province', 'latitude', 'longitude', 'newDevelopment', 'status', 'url']


,snapshot_date,city,operation,propertyCode,price,size_m2,price_m2,rooms,bathrooms,floor,exterior,district,municipality,neighborhood,province,latitude,longitude,newDevelopment,status,url
0,2026-06-20,madrid,sale,107795847,1160000.0,218.0,5321.10,3,3,4,True,Hortaleza,Madrid,Virgen del Cortijo - Manoteras,Madrid,40.483829,-3.668391,False,good,https://www.idealista.com/inmueble/107795847/
1,2026-06-20,madrid,sale,110744356,595000.0,48.0,12395.83,2,1,1,False,Barrio de Salamanca,Madrid,Lista,Madrid,40.430963,-3.671766,False,good,https://www.idealista.com/inmueble/110744356/
2,2026-06-20,madrid,sale,109947740,2100000.0,137.0,15328.47,3,3,3,True,Barrio de Salamanca,Madrid,Goya,Madrid,40.422605,-3.677340,False,good,https://www.idealista.com/inmueble/109947740/


Celda 4 — Info básica

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   snapshot_date   700 non-null    str    
 1   city            700 non-null    str    
 2   operation       700 non-null    str    
 3   propertyCode    700 non-null    str    
 4   price           700 non-null    float64
 5   size_m2         700 non-null    float64
 6   price_m2        700 non-null    float64
 7   rooms           700 non-null    int64  
 8   bathrooms       700 non-null    int64  
 9   floor           602 non-null    str    
 10  exterior        620 non-null    object 
 11  district        689 non-null    str    
 12  municipality    700 non-null    str    
 13  neighborhood    616 non-null    str    
 14  province        700 non-null    str    
 15  latitude        700 non-null    float64
 16  longitude       700 non-null    float64
 17  newDevelopment  700 non-null    bool   
 18  s

Celda 5 — Distribución por ciudad y operación

In [4]:
resumen = df.groupby(["city", "operation"]).agg(
    n_anuncios     = ("propertyCode", "count"),
    precio_medio   = ("price",    "mean"),
    precio_m2_medio= ("price_m2", "mean"),
    precio_m2_mediana=("price_m2","median"),
).round(2)

print(resumen.to_string())

                     n_anuncios  precio_medio  precio_m2_medio  precio_m2_mediana
city      operation                                                              
barcelona rent               50       2865.08            32.60              28.15
          sale               50    1727625.34          8799.36            8893.86
bilbao    rent               50       1391.66            17.69              16.19
          sale               50     921560.00          5372.77            5228.23
madrid    rent               50       2959.30            27.41              26.57
          sale               50    2046240.00          9231.55            8770.83
malaga    rent               50       1754.32            19.64              17.98
          sale               50    1173742.98          5910.50            5489.43
sevilla   rent               50       1034.46            13.16              13.28
          sale               50     800803.98          4375.40            4002.62
valencia  rent  

Celda 6 — Nulos y cobertura

In [5]:
nulos = df.isnull().sum()
pct   = (nulos / len(df) * 100).round(1)
pd.DataFrame({"nulos": nulos, "pct_%": pct}).query("nulos > 0").sort_values("pct_%", ascending=False)

,nulos,pct_%
floor,98,14.0
neighborhood,84,12.0
exterior,80,11.4
district,11,1.6


Celda 7 — Estadísticas de precio/m²

In [6]:
df.groupby("operation")["price_m2"].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
operation,,,,,,,,
rent,350.0,20.71,11.03,5.45,13.49,17.64,24.58,80.83
sale,350.0,5939.25,3052.07,672.51,3709.46,5462.57,7730.50,19000.00


Celda 8 — Resumen del snapshot por ciudad (del JSON original)

In [7]:
for ciudad, ops in raw.get("ciudades", {}).items():
    for op, stats in ops.items():
        if isinstance(stats, dict) and "error" not in stats:
            print(f"  {ciudad:12} {op:5} → mercado total: {stats.get('total_anuncios_mercado'):>6} | "
                  f"capturados: {stats.get('anuncios_capturados'):>3} | "
                  f"precio medio: {stats.get('precio_medio_m2'):>7} €/m²")

  madrid       sale  → mercado total:  17659 | capturados:  50 | precio medio: 9231.55 €/m²
  madrid       rent  → mercado total:  10858 | capturados:  50 | precio medio:   27.41 €/m²
  barcelona    sale  → mercado total:  20237 | capturados:  50 | precio medio: 8799.36 €/m²
  barcelona    rent  → mercado total:   3502 | capturados:  50 | precio medio:    32.6 €/m²
  valencia     sale  → mercado total:   8974 | capturados:  50 | precio medio: 5407.23 €/m²
  valencia     rent  → mercado total:   3241 | capturados:  50 | precio medio:   21.08 €/m²
  sevilla      sale  → mercado total:   6101 | capturados:  50 | precio medio:  4375.4 €/m²
  sevilla      rent  → mercado total:   1483 | capturados:  50 | precio medio:   13.16 €/m²
  zaragoza     sale  → mercado total:   2211 | capturados:  50 | precio medio: 2477.95 €/m²
  zaragoza     rent  → mercado total:    371 | capturados:  50 | precio medio:   13.41 €/m²
  malaga       sale  → mercado total:   6172 | capturados:  50 | precio medio: 5

In [8]:
#============================================
# Celda final — Guardar parquet raw
#============================================
import os

os.makedirs(str(ROOT / 'output/idealista/01-raw'), exist_ok=True)

ruta = str(ROOT / 'output/idealista/01-raw/raw_idealista_anuncios.parquet')
df.to_parquet(ruta, index=False)

print(f"✅ Guardado: output/idealista/01-raw/raw_idealista_anuncios.parquet")
print(f"   Filas   : {len(df)}")
print(f"   Columnas: {len(df.columns)}")
print(f"   Nulos % : {round(df.isnull().sum().sum()/df.size*100,1)}%")
print(f"   Cities  : {sorted(df['city'].unique())}")
print(f"   Ops     : {sorted(df['operation'].unique())}")
print(f"\n🎯 Listo para 02_idealista.ipynb")

✅ Guardado: output/idealista/01-raw/raw_idealista_anuncios.parquet
   Filas   : 700
   Columnas: 20
   Nulos % : 2.0%
   Cities  : ['barcelona', 'bilbao', 'madrid', 'malaga', 'sevilla', 'valencia', 'zaragoza']
   Ops     : ['rent', 'sale']

🎯 Listo para 02_idealista.ipynb


In [1]:
#============================================
# DIAGNÓSTICO — ¿Qué datos tenemos realmente?
#============================================
from pathlib import Path
import pandas as pd

ROOT = Path('/workspaces/TFG_Spain-s_Migratory_Flow')

# Buscar TODOS los archivos de datos relacionados con idealista
print("📁 ARCHIVOS IDEALISTA ENCONTRADOS:")
print("="*60)
for ext in ['*.parquet', '*.csv', '*.json', '*.xlsx']:
    for f in ROOT.rglob(ext):
        if 'idealista' in f.name.lower() or 'idealista' in str(f).lower():
            size_mb = f.stat().st_size / 1024 / 1024
            print(f"  {size_mb:6.2f} MB  →  {f.relative_to(ROOT)}")

print("\n📁 ESTRUCTURA COMPLETA output/idealista:")
print("="*60)
idealista_dir = ROOT / 'output' / 'idealista'
if idealista_dir.exists():
    for f in sorted(idealista_dir.rglob('*')):
        if f.is_file():
            size_mb = f.stat().st_size / 1024 / 1024
            print(f"  {size_mb:6.2f} MB  →  {f.relative_to(ROOT)}")
else:
    print("  ⚠️  Carpeta output/idealista NO existe")

print("\n📁 ESTRUCTURA COMPLETA output/:")
print("="*60)
for f in sorted((ROOT / 'output').rglob('*')):
    if f.is_file():
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"  {size_mb:6.2f} MB  →  {f.relative_to(ROOT)}")

📁 ARCHIVOS IDEALISTA ENCONTRADOS:
    0.05 MB  →  output/idealista/01-raw/raw_idealista_anuncios.parquet
    0.06 MB  →  output/idealista/02-clean/clean_idealista_anuncios.parquet
    0.00 MB  →  output/idealista/03-viz/insights_precio_provincia.csv
    0.00 MB  →  data/idealista_requests_log.json
    0.41 MB  →  data/idealista/2026/06/20_06_2026.json

📁 ESTRUCTURA COMPLETA output/idealista:
    0.05 MB  →  output/idealista/01-raw/raw_idealista_anuncios.parquet
    0.06 MB  →  output/idealista/02-clean/clean_idealista_anuncios.parquet
    0.00 MB  →  output/idealista/03-viz/insights_precio_provincia.csv
    0.13 MB  →  output/idealista/03-viz/viz10_dashboard_resumen.png
    0.11 MB  →  output/idealista/03-viz/viz1_distribucion_precios.png
    0.06 MB  →  output/idealista/03-viz/viz2_precio_por_provincia.png
    0.12 MB  →  output/idealista/03-viz/viz3_precio_m2_vs_tamanio.png
    0.09 MB  →  output/idealista/03-viz/viz4_heatmap_provincia_tipo.png
    0.09 MB  →  output/idealista/03-viz